In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



## BB84 Protocol Simulation (Without Attacker)

### Alice's Preparations

In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator

def get_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0])

In [4]:
KEY_LENGTH = 20
alice_bits = [get_random_bit() for _ in range(KEY_LENGTH)]
alice_bases = [get_random_bit() for _ in range(KEY_LENGTH)]

print(f"Alice's generated bits: {alice_bits}")
print(f"Alice's chosen bases (0=Z, 1=X): {alice_bases}")

Alice's generated bits: [0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1]
Alice's chosen bases (0=Z, 1=X): [0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]


### Alice Encodes Qubits

In [5]:
from qiskit import QuantumCircuit

alice_qubits = []
for i in range(KEY_LENGTH):
    qc = QuantumCircuit(1, 1)
    bit = alice_bits[i]
    basis = alice_bases[i]

    if bit == 1:
        qc.x(0)

    if basis == 1:
        qc.h(0)

    alice_qubits.append(qc)

print(f"Alice has prepared {len(alice_qubits)} qubits for sending.")

Alice has prepared 20 qubits for sending.


### Bob's Actions

In [6]:
bob_bases = [get_random_bit() for _ in range(KEY_LENGTH)]

print(f"Bob's chosen bases (0=Z, 1=X): {bob_bases}")

Bob's chosen bases (0=Z, 1=X): [0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1]


In [7]:
from qiskit.providers.basic_provider import BasicSimulator
from qiskit import transpile

bob_measurements = []
simulator = BasicSimulator()

for i in range(KEY_LENGTH):
    qc_bob = alice_qubits[i].copy()
    basis = bob_bases[i]

    if basis == 1:
        qc_bob.h(0)

    qc_bob.measure(0, 0)

    compiled_circuit = transpile(qc_bob, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc_bob)
    bob_measurements.append(int(list(counts.keys())[0]))

print(f"Bob's measurements: {bob_measurements}")

Bob's measurements: [0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0]


### Sifting and Key Generation

In [8]:
sifted_alice_key = []
sifted_bob_key = []

for i in range(KEY_LENGTH):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_measurements[i])

print(f"Sifted Alice's key: {sifted_alice_key}")
print(f"Sifted Bob's key: {sifted_bob_key}")

Sifted Alice's key: [0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0]
Sifted Bob's key: [0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0]


### Key Reconciliation and Error Checking

In [9]:
errors = 0
for i in range(len(sifted_alice_key)):
    if sifted_alice_key[i] != sifted_bob_key[i]:
        errors += 1

error_rate = errors / len(sifted_alice_key) if len(sifted_alice_key) > 0 else 0

print(f"Number of matching bits in sifted key: {len(sifted_alice_key)}")
print(f"Number of errors (disrupted qubits): {errors}")
print(f"Error rate: {error_rate:.2f}")

if errors == 0:
    print("\nBB84 Protocol successful! Alice and Bob share an identical key, and no attacker was detected.")
    final_key = sifted_alice_key
    print(f"Final shared key: {final_key}")
else:
    print("\nErrors detected! There might have been an eavesdropper or channel noise.")
    print("Alice and Bob discard the key and restart the protocol.")

Number of matching bits in sifted key: 12
Number of errors (disrupted qubits): 0
Error rate: 0.00

BB84 Protocol successful! Alice and Bob share an identical key, and no attacker was detected.
Final shared key: [0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0]
